# Mission 02: Tipping Geography Story

A newsroom wants a short visual story: **"Where in New York do taxi passengers tip the most?"** It will be published on the open web, so it must be accurate, readable and privacy-safe.

**Lessons used:** 05 (maps), 06 (storyboarding), 08 (interactive charts, optional), 16 (publishing and privacy), 17 (QA).

**Data:** NYC yellow taxi trips, February 2024 (card trips only for tips: cash tips are not recorded), and the TLC taxi zone boundaries.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets

COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance",
        "fare_amount", "tip_amount", "total_amount", "payment_type"]

def load_month(month: str) -> pd.DataFrame:
    df = pd.read_parquet(datasets.taxi(month), columns=COLS)
    start = pd.Timestamp(month + "-01")
    dur = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    ok = (df["tpep_pickup_datetime"].between(start, start + pd.offsets.MonthBegin(1), inclusive="left")
          & dur.between(1, 180) & df["trip_distance"].between(0.1, 100) & df["fare_amount"].between(3, 500))
    return df[ok].assign(duration_min=dur[ok])

zones = pd.read_csv(datasets.fetch("taxi_zone_lookup.csv"))
trips = load_month("2024-02")
trips["borough"] = trips["PULocationID"].map(zones.set_index("LocationID")["Borough"])

import geopandas as gpd

gdf = gpd.read_file(f"zip://{datasets.fetch('taxi_zones.zip')}!taxi_zones/taxi_zones.shp")
gdf["area_km2"] = gdf.geometry.area * 0.3048 ** 2 / 1e6
card = trips[trips["payment_type"] == 1]
print(f"{len(trips):,} trips, {len(card):,} card trips, {len(gdf)} zones")

## Tasks

1. **Metric.** Define the tip metric (per zone): which trips, which formula, and the minimum number of trips for a zone to be shown.
2. **Map.** Draw a choropleth of your metric by pickup zone. Choose and justify the colour scheme and the classification.
3. **Story.** Build a storyboard of 4 panels: the map, the top and bottom zones, a possible explanation (for example trip length or airport trips), and a caveat.
4. **Privacy.** Before publishing, suppress zones with too few trips. Report how many zones and trips are affected.
5. **Publish.** Export the map as PNG (for the article) and as an interactive HTML file (Folium or Plotly). Write alt text for the PNG.

In [ ]:
# Tasks 1-5
...

## Submission Checklist

- [ ] The metric uses card trips only and has a minimum sample size.
- [ ] The map uses a sequential palette and states its classification.
- [ ] The story has a caveat about what the data cannot show.
- [ ] Small zones are suppressed; exported files and alt text exist.

A solution is in `02-Tipping-Geography-Story-solution.ipynb`. Try the mission first.